In [1]:
%load_ext autoreload
import os
import sys

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, project_root)

In [2]:
import datetime as dt
from itertools import product

import ipywidgets as widgets
import pandas as pd
import plotly.graph_objects as go
import yaml
from ipywidgets import interact

from src.data.download_yfinance import download_ticker_data

### Load data

In [3]:
# declare period to load into notebook 
# Note: yfinance will have freshest data every morning CEST
params = {
    'start_date': '2022-01-01',
    'end_date': dt.datetime.today()
}

In [4]:
# declare tickers + tickers sanity test
with open("../config/tickers_list.yaml", "r") as f:
    tickers = yaml.safe_load(f)["tickers"]

tickers[:5]

['AAPL', 'MSFT', 'GDX', 'IONQ', '7974.T']

In [5]:
intervals = ["1d", "1wk", "1mo"]
all_dfs = []

for ticker, interval in product(tickers, intervals):
    df_i = download_ticker_data(
        tickers=[ticker],
        start=params["start_date"],
        end=params["end_date"],
        interval=interval,
    )
    if df_i is not None and not df_i.empty:
        df_i["ticker"] = ticker
        df_i["interval"] = interval
        all_dfs.append(df_i)

df_all = pd.concat(all_dfs)

### Basic checks

In [6]:
print(f'Number of rows and columns: {df_all.shape[0], df_all.shape[1]}')

Number of rows and columns: (34695, 8)


In [7]:
print(f'data starts: {df_all.date.min()}')
print(f'data ends: end_date = {df_all.date.max()}')

data starts: 2022-01-01 00:00:00
data ends: end_date = 2025-06-30 00:00:00


### Plot options

In [8]:
ticker = widgets.Dropdown(options=sorted(tickers), description="Ticker")
rolling = widgets.IntSlider(value=0, min=0, max=90, step=5, description="Rolling Avg")
interval = widgets.RadioButtons(options=["1d", "1wk", "1mo"], description="Interval")

# Dashboards
## Plain View

In [9]:
@interact
def plot_seasonality(ticker=ticker, interval=interval, rolling=rolling):
    df_filtered = df_all[
        (df_all["ticker"] == ticker) & (df_all["interval"] == interval)
        ].copy()
    df_filtered["date"] = pd.to_datetime(df_filtered["date"])

    if df_filtered.empty:
        print("No data available.")
        return

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=df_filtered["date"], 
        y=df_filtered["close"], 
        name="Close Price"
    ))

    if rolling > 0:
        df_filtered["rolling"] = df_filtered["close"].rolling(rolling).mean()
        fig.add_trace(go.Scatter(
            x=df_filtered["date"], 
            y=df_filtered["rolling"], 
            name=f"{rolling}-day MA"
        ))

    fig.update_layout(
        title=f"{ticker} — {interval} Value",
        xaxis_title="Date",
        yaxis_title="Price",
        template="plotly_white",
        legend=dict(x=0, y=1.1, orientation="h")
    )
    fig.show()

interactive(children=(Dropdown(description='Ticker', options=('7974.T', 'AAPL', 'AVAV', 'BB', 'BMBL', 'CGW', '…

## Candlestick

In [10]:
@interact(
    ticker=widgets.Dropdown(options=sorted(tickers), description="Ticker"),
    interval=widgets.RadioButtons(options=["1d", "1wk", "1mo"], description="Interval")
)
def plot_candlestick(ticker, interval):
    df_filtered = df_all[
        (df_all["ticker"] == ticker) & (df_all["interval"] == interval)
        ].copy()

    if df_filtered.empty:
        print("No data available.")
        return

    df_filtered = df_filtered.sort_values("date")
    df_filtered["date"] = pd.to_datetime(df_filtered["date"])
    df_filtered["ma20"] = df_filtered["close"].rolling(20).mean()
    df_filtered["ma50"] = df_filtered["close"].rolling(50).mean()

    fig = go.Figure()

    # candlestick
    fig.add_trace(go.Candlestick(
        x=df_filtered["date"],
        open=df_filtered["open"],
        high=df_filtered["high"],
        low=df_filtered["low"],
        close=df_filtered["close"],
        name="Price"
    ))

    # moving averages
    fig.add_trace(go.Scatter(
        x=df_filtered["date"], 
        y=df_filtered["ma20"], 
        mode="lines", 
        name="20-day MA"
    ))
    
    fig.add_trace(go.Scatter(
        x=df_filtered["date"], 
        y=df_filtered["ma50"], 
        mode="lines", 
        name="50-day MA"
    ))

    fig.update_layout(
        title=f"{ticker} — {interval} Candlestick Chart",
        xaxis_title="Date",
        yaxis_title="Price",
        template="plotly_dark",
        xaxis_rangeslider_visible=False,
        height=600
    )

    # volume bar chart (right)
    fig.add_trace(go.Bar(
        x=df_filtered["date"],
        y=df_filtered["volume"],
        name="Volume",
        marker_color="rgba(128,128,128,0.4)",
        yaxis="y2",
        opacity=0.5
    ))

    fig.update_layout(
        yaxis2=dict(
            overlaying="y",
            side="right",
            showgrid=False,
            title="Volume",
            anchor="x"
        )
    )

    fig.show()

interactive(children=(Dropdown(description='Ticker', options=('7974.T', 'AAPL', 'AVAV', 'BB', 'BMBL', 'CGW', '…